# Instalacion de paquetes

In [0]:
%restart_python

Url del dataset

`# https://archive.ics.uci.edu/dataset/60/automobile`

In [0]:
# para disponer de los datos de entrenamiento

!pip install ucimlrepo

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.1/300.1 MB 130.4 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


# Importar librerias

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
import statsmodels.stats.api as sms
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler

import mlflow
import os
import shutil
import importlib

from scipy import stats
from ucimlrepo import fetch_ucirepo

import warnings
warnings.filterwarnings('ignore')

# Recuperar el dataset

In [0]:
# fetch dataset 
automobile = fetch_ucirepo(id=10) 
  
# data (as pandas dataframes) 
X = automobile.data.features 

In [0]:
# borrar la variable con muchos nulos

X.drop(['normalized-losses'], axis=1, inplace=True)

In [0]:
# borrar todas las filas que contengan al menos un valor nulo

X = X.dropna()

In [0]:
X

,price,highway-mpg,city-mpg,peak-rpm,horsepower,compression-ratio,stroke,bore,fuel-system,engine-size,num-of-cylinders,engine-type,curb-weight,height,width,length,wheel-base,engine-location,drive-wheels,body-style,num-of-doors,aspiration,fuel-type,make
0,13495.0,27,21,5000.0,111.0,9.0,2.68,3.47,mpfi,130,4,dohc,2548,48.8,64.1,168.8,88.6,front,rwd,convertible,2.0,std,gas,alfa-romero
1,16500.0,27,21,5000.0,111.0,9.0,2.68,3.47,mpfi,130,4,dohc,2548,48.8,64.1,168.8,88.6,front,rwd,convertible,2.0,std,gas,alfa-romero
2,16500.0,26,19,5000.0,154.0,9.0,3.47,2.68,mpfi,152,6,ohcv,2823,52.4,65.5,171.2,94.5,front,rwd,hatchback,2.0,std,gas,alfa-romero
3,13950.0,30,24,5500.0,102.0,10.0,3.40,3.19,mpfi,109,4,ohc,2337,54.3,66.2,176.6,99.8,front,fwd,sedan,4.0,std,gas,audi
4,17450.0,22,18,5500.0,115.0,8.0,3.40,3.19,mpfi,136,5,ohc,2824,54.3,66.4,176.6,99.4,front,4wd,sedan,4.0,std,gas,audi
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,16845.0,28,23,5400.0,114.0,9.5,3.15,3.78,mpfi,141,4,ohc,2952,55.5,68.9,188.8,109.1,front,rwd,sedan,4.0,std,gas,volvo
201,19045.0,25,19,5300.0,160.0,8.7,3.15,3.78,mpfi,141,4,ohc,3049,55.5,68.8,188.8,109.1,front,rwd,sedan,4.0,turbo,gas,volvo
202,21485.0,23,18,5500.0,134.0,8.8,2.87,3.58,mpfi,173,6,ohcv,3012,55.5,68.9,188.8,109.1,front,rwd,sedan,4.0,std,gas,volvo
203,22470.0,27,26,4800.0,106.0,23.0,3.40,3.01,idi,145,6,ohc,3217,55.5,68.9,188.8,109.1,front,rwd,sedan,4.0,turbo,diesel,volvo


# Final vars

In [0]:
list_inter_ = ['wheel-base', 'horsepower', 'width', 'bore']

In [0]:
list_union_ = ['num-of-cylinders', 'engine-size', 'highway-mpg', 'peak-rpm', 'height', 'stroke', 'compression-ratio', 'horsepower', 'num-of-doors', 'wheel-base', 'bore']

In [0]:
model_name = 'automoviles-upb-2026-v2'

# Multiple linear regression

In [0]:
def load_experiment(dict_parameters, dict_metrics, model, model_name, input_sample):

    # mlflow.set_experiment(experiment_name)

    with mlflow.start_run() as run:

        # load parameters

        mlflow.log_params(dict_parameters)

        # load metrics

        for key, value in dict_metrics.items():
            mlflow.log_metric(key, value)

        # load artifacts

        for folder_ in ['eda_artifacts','relation_x_and_y','relation_x_and_y_categorical','feature_selection']:
            mlflow.log_artifacts(folder_, folder_)

        # load model

        mlflow.sklearn.log_model(
            sk_model = model,
            artifact_path = "model",
            registered_model_name = model_name, 
            input_example = input_sample
        )


In [0]:
from functions.ml_regression_functions import *

In [0]:
import functions.ml_regression_functions
importlib.reload(functions.ml_regression_functions)

<module 'functions.ml_regression_functions' from '/Workspace/Users/mangekyo.jhoan@gmail.com/Clase 5- UPB - 2026 V2/functions/ml_regression_functions.py'>

In [0]:
X

,price,highway-mpg,city-mpg,peak-rpm,horsepower,compression-ratio,stroke,bore,fuel-system,engine-size,num-of-cylinders,engine-type,curb-weight,height,width,length,wheel-base,engine-location,drive-wheels,body-style,num-of-doors,aspiration,fuel-type,make
0,13495.0,27,21,5000.0,111.0,9.0,2.68,3.47,mpfi,130,4,dohc,2548,48.8,64.1,168.8,88.6,front,rwd,convertible,2.0,std,gas,alfa-romero
1,16500.0,27,21,5000.0,111.0,9.0,2.68,3.47,mpfi,130,4,dohc,2548,48.8,64.1,168.8,88.6,front,rwd,convertible,2.0,std,gas,alfa-romero
2,16500.0,26,19,5000.0,154.0,9.0,3.47,2.68,mpfi,152,6,ohcv,2823,52.4,65.5,171.2,94.5,front,rwd,hatchback,2.0,std,gas,alfa-romero
3,13950.0,30,24,5500.0,102.0,10.0,3.40,3.19,mpfi,109,4,ohc,2337,54.3,66.2,176.6,99.8,front,fwd,sedan,4.0,std,gas,audi
4,17450.0,22,18,5500.0,115.0,8.0,3.40,3.19,mpfi,136,5,ohc,2824,54.3,66.4,176.6,99.4,front,4wd,sedan,4.0,std,gas,audi
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,16845.0,28,23,5400.0,114.0,9.5,3.15,3.78,mpfi,141,4,ohc,2952,55.5,68.9,188.8,109.1,front,rwd,sedan,4.0,std,gas,volvo
201,19045.0,25,19,5300.0,160.0,8.7,3.15,3.78,mpfi,141,4,ohc,3049,55.5,68.8,188.8,109.1,front,rwd,sedan,4.0,turbo,gas,volvo
202,21485.0,23,18,5500.0,134.0,8.8,2.87,3.58,mpfi,173,6,ohcv,3012,55.5,68.9,188.8,109.1,front,rwd,sedan,4.0,std,gas,volvo
203,22470.0,27,26,4800.0,106.0,23.0,3.40,3.01,idi,145,6,ohc,3217,55.5,68.9,188.8,109.1,front,rwd,sedan,4.0,turbo,diesel,volvo


## Interseccion

In [0]:
list_inter_

['wheel-base', 'horsepower', 'width', 'bore']

![](https://imgs.search.brave.com/l38IG6o84yDnG0myfcM_RMs3pAzS06T7o2XZTHrkN2g/rs:fit:860:0:0:0/g:ce/aHR0cHM6Ly9tZWRp/YS5nZWVrc2Zvcmdl/ZWtzLm9yZy93cC1j/b250ZW50L3VwbG9h/ZHMvMjAyMzA2MDgx/MDIxMTQvRnVuY3Rp/b24tRG9tYWluLWFu/ZC1SYW5nZS5wbmc)

In [0]:
# interseccion

poly_model_f, dict_metrics_f, X_zip_f, y_zip_f, dict_parameters_f = functions.ml_regression_functions.get_simple_linear_regression_model(
    X[list_inter_], # X
    X['price'], # y
    degree = 4,
    include_bias = False,
    interaction_only = False,
    test_size = 0.25
    )

input feature size (m): 4
output feature size (k): 69 (degree=4, include_bias=False, interaction_only =False)


In [0]:
# interseccion

dict_parameters_f['model_type'] = 'multiple_linear_regression'
dict_parameters_f['f_eng_type'] = 'interseccion'

In [0]:
dict_parameters_f

{'n_input_features': 4,
 'n_output_features': 69,
 'degree': 4,
 'include_bias': False,
 'interaction_only': False,
 'test_size': 0.25,
 'model_type': 'multiple_linear_regression',
 'f_eng_type': 'interseccion'}

In [0]:
dict_metrics_f

{'r2_train': 0.9721295395145473,
 'mae_train': 862.5453636114382,
 'mse_train': 1546313.2754073779,
 'rmse_train': 1243.5084540956598,
 'r2_test': -6.349378422142591,
 'mae_test': 7605.122053684629,
 'mse_test': 685272814.2785528,
 'rmse_test': 26177.715986666077}

In [0]:
gap_r2 = dict_metrics_f['r2_train'] - dict_metrics_f['r2_test']

In [0]:
dict_metrics_f['gap_r2'] = gap_r2

In [0]:
load_experiment(dict_parameters = dict_parameters_f,
                dict_metrics = dict_metrics_f,
                model = poly_model_f,
                model_name = model_name, # contenedor
                input_sample = X[list_inter_].iloc[[0]] ) # cambiar list_inter_

Registered model 'automoviles-upb-2026-v2' already exists. Creating a new version of this model...
Created version '4' of model 'workspace.default.automoviles-upb-2026-v2'.


## Union

In [0]:
# union

poly_model_f, dict_metrics_f, X_zip_f, y_zip_f, dict_parameters_f = functions.ml_regression_functions.get_simple_linear_regression_model(
    X[list_union_],
    X['price'],
    degree = 2,
    include_bias = False,
    interaction_only = False,
    test_size = 0.25
    )

input feature size (m): 11
output feature size (k): 77 (degree=2, include_bias=False, interaction_only =False)


In [0]:
# interseccion

dict_parameters_f['model_type'] = 'multiple_linear_regression'
dict_parameters_f['f_eng_type'] = 'union'

INFO:py4j.clientserver:Received command c on object id p0


In [0]:
gap_r2 = dict_metrics_f['r2_train'] - dict_metrics_f['r2_test']
dict_metrics_f['gap_r2'] = gap_r2

In [0]:
load_experiment(dict_parameters = dict_parameters_f,
                dict_metrics = dict_metrics_f,
                model = poly_model_f,
                model_name = model_name,
                input_sample = X[list_union_].iloc[[0]] ) # cambiar list_union_

Registered model 'automoviles-upb-2026-v2' already exists. Creating a new version of this model...
Created version '6' of model 'workspace.default.automoviles-upb-2026-v2'.


## Todas las variables numericas

In [0]:
# filtrar a las variables numericas

list_vars_to_check = X.drop('price', axis = 1).select_dtypes(include=np.number).columns.tolist() 

In [0]:
# todas

poly_model_f, dict_metrics_f, X_zip_f, y_zip_f, dict_parameters_f = functions.ml_regression_functions.get_simple_linear_regression_model(
    X[list_vars_to_check],
    X['price'],
    degree = 1,
    include_bias = False,
    interaction_only = False,
    test_size = 0.25
    )

input feature size (m): 15
output feature size (k): 15 (degree=1, include_bias=False, interaction_only =False)


In [0]:
# todas

dict_parameters_f['model_type'] = 'multiple_linear_regression'
dict_parameters_f['f_eng_type'] = 'todas_numericas'

In [0]:
gap_r2 = dict_metrics_f['r2_train'] - dict_metrics_f['r2_test']
dict_metrics_f['gap_r2'] = gap_r2

In [0]:
load_experiment(dict_parameters = dict_parameters_f,
                dict_metrics = dict_metrics_f,
                model = poly_model_f,
                model_name = model_name,
                input_sample = X[list_vars_to_check].iloc[[0]] ) # cambiar list_vars_to_check

Registered model 'automoviles-upb-2026-v2' already exists. Creating a new version of this model...
Created version '7' of model 'workspace.default.automoviles-upb-2026-v2'.


## Todas las variables incluyendo categoricas transformadas

In [0]:
df_all_x = X.drop('price', axis = 1)

In [0]:
df_all_x.select_dtypes(exclude=np.number).nunique().sort_values()

engine-location     2
aspiration          2
fuel-type           2
drive-wheels        3
engine-type         5
body-style          5
fuel-system         7
make               21
dtype: int64

![](https://imgs.search.brave.com/H7FRxTwdhqC_11iMDjLJjUj5B-Bf5kIaBFmFIANPufc/rs:fit:860:0:0:0/g:ce/aHR0cHM6Ly9zaGFy/cHNpZ2h0LmFpL3dw/LWNvbnRlbnQvdXBs/b2Fkcy8yMDIyLzA0/L3BhbmRhcy1nZXRk/dW1taWVzX3NpbXBs/ZS12aXN1YWwtZXhh/bXBsZS5wbmc)

In [0]:
X_all = pd.get_dummies(df_all_x, drop_first=True, dtype=float)

In [0]:
df_all_x.shape, X_all.shape

((193, 23), (193, 54))

In [0]:
X_all

,highway-mpg,city-mpg,peak-rpm,horsepower,compression-ratio,stroke,bore,engine-size,num-of-cylinders,curb-weight,height,width,length,wheel-base,num-of-doors,fuel-system_2bbl,fuel-system_idi,fuel-system_mfi,fuel-system_mpfi,fuel-system_spdi,fuel-system_spfi,engine-type_l,engine-type_ohc,engine-type_ohcf,engine-type_ohcv,engine-location_rear,drive-wheels_fwd,drive-wheels_rwd,body-style_hardtop,body-style_hatchback,body-style_sedan,body-style_wagon,aspiration_turbo,fuel-type_gas,make_audi,make_bmw,make_chevrolet,make_dodge,make_honda,make_isuzu,make_jaguar,make_mazda,make_mercedes-benz,make_mercury,make_mitsubishi,make_nissan,make_peugot,make_plymouth,make_porsche,make_saab,make_subaru,make_toyota,make_volkswagen,make_volvo
0,27,21,5000.0,111.0,9.0,2.68,3.47,130,4,2548,48.8,64.1,168.8,88.6,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,27,21,5000.0,111.0,9.0,2.68,3.47,130,4,2548,48.8,64.1,168.8,88.6,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,26,19,5000.0,154.0,9.0,3.47,2.68,152,6,2823,52.4,65.5,171.2,94.5,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,30,24,5500.0,102.0,10.0,3.40,3.19,109,4,2337,54.3,66.2,176.6,99.8,4.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,22,18,5500.0,115.0,8.0,3.40,3.19,136,5,2824,54.3,66.4,176.6,99.4,4.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,28,23,5400.0,114.0,9.5,3.15,3.78,141,4,2952,55.5,68.9,188.8,109.1,4.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
201,25,19,5300.0,160.0,8.7,3.15,3.78,141,4,3049,55.5,68.8,188.8,109.1,4.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
202,23,18,5500.0,134.0,8.8,2.87,3.58,173,6,3012,55.5,68.9,188.8,109.1,4.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
203,27,26,4800.0,106.0,23.0,3.40,3.01,145,6,3217,55.5,68.9,188.8,109.1,4.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [0]:
# todas, incluidas categoricas

poly_model_f, dict_metrics_f, X_zip_f, y_zip_f, dict_parameters_f = functions.ml_regression_functions.get_simple_linear_regression_model(
    X_all,
    X['price'],
    degree = 1,
    include_bias = False,
    interaction_only = False,
    test_size = 0.25
    )

input feature size (m): 54
output feature size (k): 54 (degree=1, include_bias=False, interaction_only =False)


In [0]:
# todas

dict_parameters_f['model_type'] = 'multiple_linear_regression'
dict_parameters_f['f_eng_type'] = 'todas_incluyendo_categoricas'

INFO:py4j.clientserver:Received command c on object id p0


In [0]:
gap_r2 = dict_metrics_f['r2_train'] - dict_metrics_f['r2_test']
dict_metrics_f['gap_r2'] = gap_r2

In [0]:
load_experiment(dict_parameters = dict_parameters_f,
                dict_metrics = dict_metrics_f,
                model = poly_model_f,
                model_name = model_name,
                input_sample = X_all.iloc[[0]] )

Registered model 'automoviles-upb-2026-v2' already exists. Creating a new version of this model...
Created version '8' of model 'workspace.default.automoviles-upb-2026-v2'.


# Random Forest

In [0]:
dict_parameters_rf = {
    'n_estimators': 101,
    'max_depth': 5,
    'max_features': 'log2'
}

In [0]:
# cambiar list_union_

model_rf, dict_metrics_rf, X_zip_rf, y_zip_rf, dict_parameters_rf = functions.ml_regression_functions.get_random_forest_regression_model(
    X[list_vars_to_check], # cambiar
    # X_all,
    X['price'],
    dict_parameters_rf, 
    degree = 1, # no disponible
    include_bias = False, # no disponible
    interaction_only = False, # no disponible
    test_size = 0.25
)

input feature size (m): 15
output feature size (k): 15 (degree=1, include_bias=False, interaction_only =False)


In [0]:
rf_ = model_rf.named_steps["random_forest"]

In [0]:
list_all_x = X_all.columns.tolist()

In [0]:
# feature importance, cambiar list_union_

df_variables_summary = pd.DataFrame({'VAR_NAME':list_vars_to_check,'WEIGHT':rf_.feature_importances_}).sort_values('WEIGHT', ascending = False) # cambiar list_union_
# df_variables_summary = pd.DataFrame({'VAR_NAME':list_all_x,'WEIGHT':rf_.feature_importances_}).sort_values('WEIGHT', ascending = False) # cambiar list_union_

In [0]:
df_variables_summary

,VAR_NAME,WEIGHT
9,curb-weight,0.221627
7,engine-size,0.170234
3,horsepower,0.115541
1,city-mpg,0.100159
0,highway-mpg,0.083122
11,width,0.071731
8,num-of-cylinders,0.070345
12,length,0.055936
13,wheel-base,0.044494
2,peak-rpm,0.021782


In [0]:
# save feature importance

os.makedirs("rf_artifacts/feature_importance", exist_ok=True)
df_variables_summary.to_csv("rf_artifacts/feature_importance/df_variables_summary.csv", index=False)

In [0]:
os.makedirs("rf_artifacts/dataset", exist_ok=True)

In [0]:
# save datasets, cambiar list_union_

for x_,y_, name in zip(X_zip_rf,y_zip_rf,['train','test']):

  y_pred = model_rf.predict(x_[list_vars_to_check]) #cambiar list_union_
  # y_pred = model_rf.predict(x_[list_all_x]) #cambiar list_union_
  x_['y_pred'] = y_pred
  df_sample = pd.concat([x_,y_], axis = 1).rename({'price':'y_true'}, axis = 1)

  df_sample['residuo'] = df_sample['y_true'] - df_sample['y_pred']
  df_sample['residuo_abs'] = df_sample['residuo'].apply(lambda x: abs(x))

  df_sample.to_csv(f"rf_artifacts/dataset/df_{name}_set.csv", index=False)

In [0]:
dict_parameters_rf['model_type'] = 'random_forest'
dict_parameters_rf['f_eng_type'] = 'todas_numericas' # cambiar

In [0]:
# evaluacion del test

df_test_ = pd.read_csv('rf_artifacts/dataset/df_test_set.csv')
mae_test = dict_metrics_rf['mae_test']

In [0]:
os.makedirs('rf_artifacts/quality_test', exist_ok=True)

In [0]:
functions.ml_regression_functions.get_predictions_quality(df_test_, 'y_true', mae_test, q = 5)

In [0]:
functions.ml_regression_functions.get_residual_graph(df_test_.y_true, df_test_.y_pred)

In [0]:
artifact_path = 'rf_artifacts'

In [0]:
gap_r2 = dict_metrics_rf['r2_train'] - dict_metrics_rf['r2_test']
dict_metrics_rf['gap_r2'] = gap_r2

In [0]:
# mlflow.set_experiment(experiment_name)

with mlflow.start_run() as run:

    # load parameters

    mlflow.log_params(dict_parameters_rf)

    # load metrics

    for key, value in dict_metrics_rf.items():
        mlflow.log_metric(key, value)

    # load artifacts EDA

    for folder_ in ['eda_artifacts','relation_x_and_y','relation_x_and_y_categorical','feature_selection']:
        mlflow.log_artifacts(folder_, folder_)

    # load artifacts

    for sub_folder in ['feature_importance','dataset','quality_test']:

        path_name = f"{artifact_path}/{sub_folder}"
        mlflow.log_artifacts(path_name, artifact_path=sub_folder)
        shutil.rmtree(path_name)
        os.makedirs(path_name, exist_ok=True) 

    # load model

    mlflow.sklearn.log_model(
        sk_model = model_rf,
        artifact_path = "model",
        registered_model_name = model_name, 
        input_example=X[list_vars_to_check].iloc[[0]]  # cambiar list_union_
        # input_example = X_all.iloc[[0]]
    )

INFO:py4j.clientserver:Received command c on object id p0
Registered model 'automoviles-upb-2026-v2' already exists. Creating a new version of this model...
Created version '10' of model 'workspace.default.automoviles-upb-2026-v2'.


# Xgboost

In [0]:
# landspace of parameters

dict_parameters_xgb ={
    'n_estimators':1200,
    'max_depth':3,
    'learning_rate':0.01,
    'test_size': 0.25,
    'validation_size': 0.2,
    'reg_lambda': 1.0,
    'reg_alpha': 0.0,
    'degree':1,
    'include_bias': False,
    'interaction_only': False
}

INFO:py4j.clientserver:Received command c on object id p0


In [0]:
list_inter_

['wheel-base', 'horsepower', 'width', 'bore']

In [0]:
model_xgb, dict_metrics_xgb, X_zip_xgb, y_zip_xgb, dict_parameters_xgb = functions.ml_regression_functions.get_xgboost_regression_model(
    X[list_inter_], # cambiar list_union_
    X['price'],
    dict_parameters_xgb
)

train size: (108, 4), validation size: (36, 4), test size: (49, 4)
train %: 0.5595854922279793, validation %: 0.18652849740932642, test %: 0.2538860103626943
input feature size (m): 4
output feature size (k): 4 (degree=1, include_bias=False, interaction_only =False)


In [0]:
len(X_zip_xgb)

3

In [0]:
dict_parameters_xgb['model_type'] = 'xgboost'
dict_parameters_xgb['f_eng_type'] = 'interseccion'

In [0]:
os.makedirs('xgb_artifacts/dataset', exist_ok=True)

In [0]:
# save datasets

for x_,y_, name in zip(X_zip_xgb,y_zip_xgb,['train','validation','test']):

  y_pred = model_xgb.predict(x_[list_inter_]) # cambiar list_union_
  x_['y_pred'] = y_pred
  df_sample = pd.concat([x_,y_], axis = 1).rename({'price':'y_true'}, axis = 1)

  df_sample['residuo'] = df_sample['y_true'] - df_sample['y_pred']
  df_sample['residuo_abs'] = df_sample['residuo'].apply(lambda x: abs(x))

  df_sample.to_csv(f"xgb_artifacts/dataset/df_{name}_set.csv", index=False)

In [0]:
os.makedirs('xgb_artifacts/quality_test', exist_ok=True)

In [0]:
# evaluacion del test

df_test_xgb = pd.read_csv('xgb_artifacts/dataset/df_test_set.csv')
mae_test_xgb = dict_metrics_xgb['mae_test']

In [0]:
functions.ml_regression_functions.get_residual_graph(df_test_xgb.y_true, df_test_xgb.y_pred, 'xgb_artifacts')

In [0]:
functions.ml_regression_functions.get_predictions_quality(df_test_xgb, 'y_true', mae_test_xgb, q = 5, path ='xgb_artifacts')

In [0]:
artifact_path = 'xgb_artifacts'

In [0]:
# mlflow.set_experiment(experiment_name)

with mlflow.start_run() as run:

    # load parameters

    mlflow.log_params(dict_parameters_xgb)

    # load metrics

    for key, value in dict_metrics_xgb.items():
        mlflow.log_metric(key, value)

    # load artifacts EDA

    for folder_ in ['eda_artifacts','relation_x_and_y','relation_x_and_y_categorical','feature_selection']:
        mlflow.log_artifacts(folder_, folder_)

    # load artifacts

    for sub_folder in ['dataset','quality_test']:

        path_name = f"{artifact_path}/{sub_folder}"
        mlflow.log_artifacts(path_name, artifact_path=sub_folder)
        shutil.rmtree(path_name)
        os.makedirs(path_name, exist_ok=True) 
        
    # load model

    mlflow.sklearn.log_model(
        sk_model = model_xgb,
        artifact_path = "model",
        registered_model_name =model_name, 
        input_example=X[list_inter_].iloc[[0]]  # cambiar list_union_
    )

INFO:py4j.clientserver:Received command c on object id p0
Registered model 'automoviles-upb-2026-v2' already exists. Creating a new version of this model...
Created version '12' of model 'workspace.default.automoviles-upb-2026-v2'.


# Ridge & Lasso

![](https://towardsdatascience.com/wp-content/uploads/2020/11/1nrWncnoJ4V_BkzEf1pd4MA.png)

![](https://imgs.search.brave.com/bZTgr2xhbJaDaO3QuOYpC7tpYLSB6kArC42tL3c0mRY/rs:fit:860:0:0:0/g:ce/aHR0cHM6Ly9jZG4u/c2hvcGFjY2luby5j/b20vaWdtZ3VydS9h/cnRpY2xlcy9oeXBl/cnBhcmFtZXRlci10/dW5pbmctMTYyMzM4/OTY2MDc0OTkxOV9s/LmpwZz92PTUzMg)

![](https://imgs.search.brave.com/m9OADIPHUZP1Vtk4Gdii1zJ1APpuxp-HTW8T1JJ167k/rs:fit:860:0:0:0/g:ce/aHR0cHM6Ly9jZG4t/aW1hZ2VzLTEubWVk/aXVtLmNvbS9tYXgv/NzQ3LzEqdE1DeGE1/bTYtUU51bHJRczI4/dkJNdy5wbmc)

# Lasso (L1)
Puede llevar coeficientes a 0 exacto

In [0]:
lasso_params = {
    "model__alpha": np.logspace(-3, 1, 9) 
}

In [0]:
model_ls, dict_metrics_ls, X_zip_ls, y_zip_ls, dict_parameters_ls = functions.ml_regression_functions.get_lasso_regression(
    X[list_union_], # cambiar list_union_
    X['price'],
    lasso_params,
    cv = 3
)

In [0]:
# save model type and number of vars

dict_parameters_ls['model_type'] = 'multiple_lasso_regression'
dict_parameters_ls['n_input_features'] = len(list_union_) # cambiar list_union_
dict_parameters_ls['preprocess_transformation'] = 'StandardScaler'

In [0]:
# mlflow.set_experiment(experiment_name)

with mlflow.start_run() as run:

    # load parameters

    mlflow.log_params(dict_parameters_ls)

    # load metrics

    for key, value in dict_metrics_ls.items():
        mlflow.log_metric(key, value)

    # load model

    mlflow.sklearn.log_model(
        sk_model = model_ls,
        artifact_path = "model",
        registered_model_name = model_name, 
        input_example=X[list_union_].iloc[[0]]  # cambiar list_union_
    )

INFO:py4j.clientserver:Received command c on object id p0
Registered model 'automoviles-upb-2026-v2' already exists. Creating a new version of this model...
Created version '13' of model 'workspace.default.automoviles-upb-2026-v2'.


# Ridge (L2)
Encoje coeficientes hacia 0 pero no los hace exactamente 0

In [0]:
ridge_params = {
    "model__alpha": np.logspace(-3, 3, 13) 
}

In [0]:
model_rd, dict_metrics_rd, X_zip_rd, y_zip_rd, dict_parameters_rd = functions.ml_regression_functions.get_ridge_regression(
    X[list_union_], # cambiar list_union_
    X['price'],
    ridge_params
)

In [0]:
# save model type and number of vars

dict_parameters_rd['model_type'] = 'multiple_ridge_regression'
dict_parameters_rd['n_input_features'] = len(list_union_) # cambiar list_union_

INFO:py4j.clientserver:Received command c on object id p0


In [0]:
dict_parameters_rd['f_eng_type'] = 'union'

In [0]:
# mlflow.set_experiment(experiment_name)

with mlflow.start_run() as run:

    # load parameters

    mlflow.log_params(dict_parameters_rd)

    # load metrics

    for key, value in dict_metrics_rd.items():
        mlflow.log_metric(key, value)

    # load model

    mlflow.sklearn.log_model(
        sk_model = model_rd,
        artifact_path = "model",
        registered_model_name = model_name, 
        input_example=X[list_union_].iloc[[0]] # cambiar list_union_
    ) 

Registered model 'automoviles-upb-2026-v2' already exists. Creating a new version of this model...
Created version '14' of model 'workspace.default.automoviles-upb-2026-v2'.
